In [1]:
%matplotlib inline
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import json
from itertools import product
import matplotlib.pyplot as plt

hytraits_path = (Path.cwd().parent/'hytraits').resolve()
if str(hytraits_path) not in sys.path:
    sys.path.append(str(hytraits_path))
import hytraits as H 
from paths import get_paths

In [2]:
run_all_cells = False

In [3]:
model_dir = Path.cwd()/'io'/'model'
ideploy_dir = Path.cwd()/'io'/'ideploy'
post_dir = Path.cwd()/'io/post/2025-11-07'
post_dir.mkdir(parents=True, exist_ok=True)

In [4]:
colors6 = ['#EF476F', '#F78C6B', '#FFD166', '#06D6A0', '#118A32', '#073B4C']

orig_to_comp = [('IC', 'ic'),
                ('NPOC', 'npoc'),
                ('Cl(-)', 'cl'), 
                ('SO4(2-)', 'so4'), 
                ('Silica', 'si'), 
                ('NO2/NO3 ', 'no23'), # note the space
                ('SRP', 'srp'),
                ('NH4(+)', 'nh4'),
                ('TN', 'tn'),
                ('TP', 'tp'),
                ('TSS', 'tss'),
                ('CF_chl', 'cfchl'),
                ('CF_PC', 'cfpc'),
                ('PC:chl', 'pcchl')]
traits = [c for (o, c) in orig_to_comp]
traits.sort() 

treatments = ['asis', 'move']

In [5]:
# standard coefficient plots
run_this_cell = True

if run_this_cell or run_all_cells:
    # traits = ['cfchl']
    
    for trait in traits:
        print(f'{trait}: stdcoeffs plot')
        fig, axs = plt.subplots(2, 1, figsize=(12, 6))
        for (i_treat, treat) in enumerate(treatments):
            model_dirs = [d for d in model_dir.glob(f'plsr__*{treat}*__{trait}*')]
            model_dirs.sort()
    
            for (i_md, md) in enumerate(model_dirs):
                model_file = md/f'home/pravindran/sophia-lakeview/io/model/{md.stem}/model.npz'
                model = H.plsr_load_model(model_file)
                axs[i_treat] = H.plot_stat_vs_wavelength(ax=axs[i_treat],
                                                         waves=model.get(key='wavelengths'),
                                                         wave_ranges=model.get(key='kept_wave_ranges'),
                                                         samples=model.get(key='std_coefficients'),
                                                         color=colors6[i_md],
                                                         show_sdev=False)
                axs[i_treat].set_title(f'{trait}:{treat}:standard coefficients')

        plt.tight_layout()
        # plt.show()
        plt.savefig(post_dir/f'{trait}_stdcoeffs.jpg')
        plt.close()

cfchl: stdcoeffs plot
cfpc: stdcoeffs plot
cl: stdcoeffs plot
ic: stdcoeffs plot
nh4: stdcoeffs plot
no23: stdcoeffs plot
npoc: stdcoeffs plot
pcchl: stdcoeffs plot
si: stdcoeffs plot
so4: stdcoeffs plot
srp: stdcoeffs plot
tn: stdcoeffs plot
tp: stdcoeffs plot
tss: stdcoeffs plot


In [6]:
# true-pred plots
run_this_cell = True

if run_this_cell or run_all_cells:
    # traits = ['cfchl']
    
    for trait in traits:
        print(f'{trait}: true-pred plot')

        fig, axs = plt.subplots(2, 6, figsize=(18, 6), sharex=True, sharey=True)
        for (i_treat, treat) in enumerate(treatments):
            ideploy_dirs = [d for d in ideploy_dir.glob(f'plsr__*{treat}*__{trait}*')]
            ideploy_dirs.sort()
            for (i_idd, idd) in enumerate(ideploy_dirs):
                pred_file = idd/f'home/pravindran/sophia-lakeview/io/ideploy/{idd.stem}/preds.csv'
                pred_df = pd.read_csv(pred_file)

                color_df = pd.DataFrame({'sample_id': pred_df['sample_id'],
                                         'color': [colors6[i_idd]]*len(pred_df['sample_id'])})

                metric_file = idd/f'home/pravindran/sophia-lakeview/io/ideploy/{idd.stem}/metrics.csv'
                metric_df = pd.read_csv(metric_file)
                r2 = np.mean(metric_df['r2'].values)
                rnrmse = np.mean(metric_df['range_normalized_rmse'].values)
                stats = (f'R2 = {r2:.2f}\n'
                         f'RNRMSE = {rnrmse:.1f}')

                axs[i_treat, i_idd] = H.plot_pred_vs_true(axs[i_treat, i_idd],
                                                          pred_df=pred_df,
                                                          color_df=color_df)
                axs[i_treat, i_idd].text(0.07, 
                                         0.85, 
                                         stats, 
                                         fontsize=9,
                                         transform=axs[i_treat, i_idd].transAxes,
                                         horizontalalignment='left')
                
        fig.suptitle(f'{trait}', y=1.0)
                
        plt.tight_layout()
        # plt.show()
        plt.savefig(post_dir/f'{trait}_true-pred.jpg')
        plt.close()

cfchl: true-pred plot
cfpc: true-pred plot
cl: true-pred plot
ic: true-pred plot
nh4: true-pred plot
no23: true-pred plot
npoc: true-pred plot
pcchl: true-pred plot
si: true-pred plot
so4: true-pred plot
srp: true-pred plot
tn: true-pred plot
tp: true-pred plot
tss: true-pred plot
